Clustering Model using K mean and DBscan

This model aims to anwser two questions users may want to find. 

1) KMeans: Which NSW Local Government Areas (LGAs) have similar road crash patterns based on the selected crash features?

2) DBSCAN: Which LGA-year records show unusual crash patterns that do not fit into any identified dense group?


Importing all needed libaries like panda and sklearn

In [ ]:
import numpy as np                                   
import pandas as pd                                  
from sklearn.cluster import KMeans, DBSCAN          
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score  

Load the cleaned dataset

In [ ]:
data = pd.read_csv("../../backend/data/lga_yearly_aggregates_clustering.csv")
print (data.head())

In [ ]:
#data = data[data["LGA"] != "Unincorporated NSW"]    # remove Unincorporated data for testing ONLY.
print(data.shape)

Select the columns for the dataset to be used

In [ ]:
Dcolumns = ["Crash rate per 100k", "Severe %", "Wet-weather %", "Rural-road %"]
X = data[Dcolumns].values

Scale the values in the dataset so it counts equally.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Creating a table to show scores for s which silhouette data

In [ ]:
kmeans_results = []

Test K-Means with 2 to 8 clusters and record the silhouette score for each, to help choose K

In [ ]:
# s = silhoutte score which then we will choose our K. In CSV kmeans_k_scores.csv  the column will be saved as k instead of s to avoid confusion
for s in range (2,9):
    range_s = KMeans (n_clusters=s, random_state=20)#K-Means with s clusters (random_state keeps the results repeatable)
    s_labels = range_s.fit_predict(X_scaled)# train it on the scaled data 
    silhouette_avg = silhouette_score(X_scaled,s_labels)
    kmeans_results.append({'k': s,'silhouette_score': silhouette_avg})# store S score in the lsit 
    print(f's = {s}  Silhouette Score: {silhouette_avg:.2f}') # Print score 

Train the Kmeans model based on the best/highest silhouette score. 

In [ ]:
# The highest score is s = 3 (0.31), so K = 3 is used to create and train the model

K=3
model = KMeans(n_clusters = K , random_state=20)
model.fit(X_scaled)


Step 7: Evalute Kmeans using Silhoutte score


In [ ]:
#first must give every row a clustering number

predictions = model.predict(X_scaled)

# Now must print one score for the whole result
silhouette_avg = silhouette_score(X_scaled, predictions)
print (f'Silhoutte Score:{silhouette_avg:.2f}')

DBscans part

Apply DBSCAN

In [ ]:
dbscan = DBSCAN(eps=1.0, min_samples=5) # set as maximun distance and set at least 5 samples 
dbscan.fit(X_scaled)#Run dbscans

row = dbscan.labels_ # make sure each row is labeled

Evaluating DBSCAN using silhoutte score

In [ ]:
silhouette_avg = silhouette_score(X_scaled, row)
print (f'Silhouette Score: {silhouette_avg:.2f}') # decimals form

Identify the noise in the dataset

In [ ]:
noise = np.sum (row ==-1 ) # showing all rows with negative 1
print(f'Number of noise points: {noise}')


Testing with diffirent values to detemine which eps is better using noise and silhouette value

In [ ]:
for eps_try in [0.5, 1.0, 1.5, 2.0]:                                 # try four eps values
    dbscan_try = DBSCAN(eps=eps_try, min_samples=5)                 
    dbscan_try.fit(X_scaled)                                        
    labels_try = dbscan_try.labels_                                  
    silhouette_avg = silhouette_score(X_scaled, labels_try)          # determining thesilhouette score 
    noise = np.sum(labels_try == -1)                               
    print(f'eps = {eps_try}  Silhouette Score: {silhouette_avg:.2f}  Number of noise points: {noise}')

Table: Clustering sizes and profiles

In [ ]:
from IPython.display import display # for borders 

data["kmean data"] = predictions # get values from kmean
data["dbscans data"] = row # adds a group number
display(data["kmean data"].value_counts()) # making sure no cluster is small or hold everything
cluster_avg = data.groupby("kmean data")[Dcolumns].mean().round(2) # getting the average for the data wihtin each cluster
display(cluster_avg)
display(data[Dcolumns].mean().round(2).to_frame("Average")) # create a another box to display the average of each column based on the clusters
display(data["dbscans data"].value_counts())

Identify which LGAs belong to each K-Means cluster based their crash patterns 

Since our K= 3 we will have 3 clusters:


Cluster 0 : Higher rural-road %, highest casualty share, and highest casualty %

In [ ]:
cluster0 = data[data["kmean data"] == 0] # finding hte once in clustering 0                # rows in cluster 0
counts0 = cluster0["LGA"].value_counts()
print("Cluster 0: Rural, highest casualty share %, lowest wet-surface %")                # years each LGA spent in cluster 0
print(counts0[counts0 == 5])    

Cluster 1 :  Mostly non-rural areas and low crash rate

In [ ]:
cluster1 = data[data["kmean data"] == 1]                 # rows in cluster 1
counts1 = cluster1["LGA"].value_counts()
print("Cluster 1: Mostly non-rural, low crash rate %")                 # years each LGA spent in cluster 1
print(counts1[counts1 == 5])     

Cluster 2: Rural areas, wet-surface share % and highest crash rate

In [ ]:
cluster2 = data[data["kmean data"] == 2]       # rows in cluster 2
counts2 = cluster2["LGA"].value_counts()# years each LGA spent in cluster 2
print("Cluster 2: Rural, highest wet-surface %, highest crash rate % ")
print(counts2[counts2 == 5])

Stating the total amount of stable LGAS in each cluster and the total amount of LGAs 

In [ ]:
print("Cluster 0 stable LGAs:", len(counts0[counts0 == 5]))     
print("Cluster 1 stable LGAs:", len(counts1[counts1 == 5]))     
print("Cluster 2 stable LGAs:", len(counts2[counts2 == 5]))     
print("Total LGAs:", data["LGA"].nunique())                     

Which LGAs are unusual? using DBscans 

In [ ]:
row_noise = data[data["dbscans data"] == -1] # get all the rows labeled as -1
print (row_noise[["LGA", "Year", "Population", "total_crashes"] + Dcolumns + ["kmean data"]]) 
print(row_noise["LGA"].value_counts())# count the total which is flaged per LGA in a year

Save all csv for visulisation

In [ ]:
import os

# CSV of the complete dataset with K-Means and DBSCAN labels
data.to_csv("../../backend/data/lga_clusters.csv", index=False)

# CSV to show the average feature values for each K-Means cluster
cluster_avg.to_csv("../../backend/data/cluster_profiles.csv")

# CSV that has k mean silhoutte scores
kmeans_scores = pd.DataFrame(kmeans_results)
kmeans_scores.to_csv("../../backend/data/kmeans_k_scores.csv", index=False)

# CSV to compare DBSCAN using eps vlaues to determine how the number of clusters and noise points chnage
epsresults =[]

for eps in [0.5, 1.0, 1.5, 2.0]:
    labels = DBSCAN(eps=eps, min_samples = 5).fit_predict(X_scaled)
    clusters = len (set(labels)- {-1})
    noise = sum(labels == -1)

    epsresults.append([eps, clusters, noise]) #place the results into this box

# Save eps results into a eps
pd.DataFrame(epsresults, columns =["eps", "cluster_count", "noise_count"]).to_csv("../../backend/data/dbscan_eps_comparison.csv", index=False)

# CSV for all unusal record
data[data["dbscans data"] == -1].to_csv("../../backend/data/dbscan_noise_records.csv", index=False)

Save model for prediction in Assignment 3
 Mainly for the prediction sciprt to load and assign new LGA record to a cluster 

 


In [ ]:
import joblib

joblib.dump({
    "scaler": scaler,
    "kmeans": model,
    "features": Dcolumns,
    "cluster_names": 
    {
        0: "Rural, highest casualty share, lowest wet-surface %",
        1: "Mostly non-rural, lowest crash rate",
        2: "Rural, highest wet-surface %, highest crash rate",
    },
}, "/reference/predict_scripts/kmeans_lga.joblib")

print("Model Saved")